# ==============================
# SPAM EMAIL DETECTION PROJECT
# ==============================

# Import Libraries


In [6]:
import pandas as pd
import numpy as np
import re
import string
import joblib

from sklearn.model_selection import train_test_split

from sklearn.feature_extraction.text import (
    TfidfVectorizer,
    ENGLISH_STOP_WORDS
)

from sklearn.naive_bayes import MultinomialNB

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

# LOAD DATASET


In [7]:
df = pd.read_csv("spam.csv", encoding='latin-1')
print(df.head())


  Category                                            Message
0      ham  Go until jurong point, crazy.. Available only ...
1      ham                      Ok lar... Joking wif u oni...
2     spam  Free entry in 2 a wkly comp to win FA Cup fina...
3      ham  U dun say so early hor... U c already then say...
4      ham  Nah I don't think he goes to usf, he lives aro...


In [8]:
# Keep required columns
df = df[['Category', 'Message']]

# Rename columns
df.columns = ['label', 'message']

# Display first rows
print(df.head())

  label                                            message
0   ham  Go until jurong point, crazy.. Available only ...
1   ham                      Ok lar... Joking wif u oni...
2  spam  Free entry in 2 a wkly comp to win FA Cup fina...
3   ham  U dun say so early hor... U c already then say...
4   ham  Nah I don't think he goes to usf, he lives aro...


# Clean Text

In [9]:
stop_words = ENGLISH_STOP_WORDS

def clean_text(text):

    # Lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\\S+', '', text)

    # Remove numbers
    text = re.sub(r'\\d+', '', text)

    # Remove punctuation
    text = text.translate(str.maketrans('', '', string.punctuation))

    # Remove spaces
    text = text.strip()

    # Remove stopwords
    text = ' '.join(
        word for word in text.split()
        if word not in stop_words
    )

    return text

# Apply cleaning
df['message'] = df['message'].apply(clean_text)

print(df.head())

  label                                            message
0   ham  jurong point crazy available bugis n great wor...
1   ham                            ok lar joking wif u oni
2  spam  free entry 2 wkly comp win fa cup final tkts 2...
3   ham                        u dun say early hor u c say
4   ham                      nah dont think goes usf lives


# Text Cleaning Function

In [10]:
stop_words = ENGLISH_STOP_WORDS

def clean_text(text):

    # Convert to lowercase
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\\S+', '', text)

    # Remove numbers
    text = re.sub(r'\\d+', '', text)

    # Remove punctuation
    text = text.translate(
        str.maketrans('', '', string.punctuation)
    )

    # Remove extra spaces
    text = text.strip()

    # Remove stopwords
    text = ' '.join(
        word for word in text.split()
        if word not in stop_words
    )

    return text

# Apply Cleaning

In [11]:
df['message'] = df['message'].apply(clean_text)

print(df.head())

  label                                            message
0   ham  jurong point crazy available bugis n great wor...
1   ham                            ok lar joking wif u oni
2  spam  free entry 2 wkly comp win fa cup final tkts 2...
3   ham                        u dun say early hor u c say
4   ham                      nah dont think goes usf lives


# Convert Labels

In [12]:
df['label'] = df['label'].map({
    'ham': 0,
    'spam': 1
})

print(df.head())

   label                                            message
0      0  jurong point crazy available bugis n great wor...
1      0                            ok lar joking wif u oni
2      1  free entry 2 wkly comp win fa cup final tkts 2...
3      0                        u dun say early hor u c say
4      0                      nah dont think goes usf lives


# Features and Target

In [13]:
X = df['message']
y = df['label']

# TF-IDF Vectorization

In [14]:
vectorizer = TfidfVectorizer()

X = vectorizer.fit_transform(X)

print(X.shape)

(5572, 9310)


# Train-Test Split

In [15]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print(X_train.shape)
print(X_test.shape)

(4457, 9310)
(1115, 9310)


# Train Model

In [17]:
model = MultinomialNB()

model.fit(X_train, y_train)

print("Model trained successfully!")

Model trained successfully!


# Predictions

In [18]:
y_pred = model.predict(X_test)

print(y_pred[:10])

[0 0 0 0 0 0 0 0 0 0]


# Accuracy & Evaluation

In [19]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:\n")
print(confusion_matrix(y_test, y_pred))

Accuracy: 0.9641255605381166

Classification Report:

              precision    recall  f1-score   support

           0       0.96      1.00      0.98       966
           1       1.00      0.73      0.84       149

    accuracy                           0.96      1115
   macro avg       0.98      0.87      0.91      1115
weighted avg       0.97      0.96      0.96      1115


Confusion Matrix:

[[966   0]
 [ 40 109]]


# Test Custom Message

In [20]:
sample = ["Congratulations! You won a free iPhone"]

sample_vector = vectorizer.transform(sample)

prediction = model.predict(sample_vector)

if prediction[0] == 1:
    print("Spam Message")
else:
    print("Not Spam")

Spam Message
